# IEEE-CIS Fraud Detection — Exploratory Data Analysis

**Dataset:** IEEE-CIS Fraud Detection (Kaggle)  
**Source tables:** `fraud_detection.transactions_joined` (BigQuery)  
**Objective:** Understand data characteristics that drive feature engineering decisions.

> All analysis logic lives in `src/features/eda.py`. This notebook is a thin orchestration layer — no pandas/numpy logic inline.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.utils.config import load_config
from src.features.eda import (
    load_from_bigquery,
    plot_class_distribution,
    plot_null_heatmap,
    plot_transaction_amount,
    plot_categorical_fraud_rates,
    plot_d_column_correlations,
)

config = load_config('dev')
print('Config loaded. Project:', config['gcp']['project_id'])

In [ ]:
# Load data from BigQuery — remove limit=None for full dataset
df = load_from_bigquery(config, limit=None)
print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
df.head(3)

## 1. Class Distribution

Understanding the fraud prevalence is the first step. High imbalance dictates metric choice, sampling strategy, and threshold selection.

In [ ]:
fig = plot_class_distribution(df)
fig.savefig('../evaluation/01_class_distribution.png', bbox_inches='tight')
fig.show()

## 2. Null Analysis

IEEE-CIS has extreme null rates in V-features (Vesta engineered, up to ~94%) and identity columns (only ~25% of transactions have identity data). This drives imputation strategy and feature selection.

In [ ]:
fig = plot_null_heatmap(df)
fig.savefig('../evaluation/02_null_heatmap.png', bbox_inches='tight')
fig.show()

In [ ]:
# Top 20 highest-null columns
null_summary = (df.isnull().mean() * 100).sort_values(ascending=False).head(20)
print(null_summary.to_string(float_format='{:.1f}%'.format))

## 3. Transaction Amount by Fraud Label

Distribution of `TransactionAmt` across fraud vs legitimate transactions reveals whether fraudsters favour specific amount ranges.

In [ ]:
fig = plot_transaction_amount(df)
fig.savefig('../evaluation/03_transaction_amount.png', bbox_inches='tight')
fig.show()

In [ ]:
# Summary stats
print(df.groupby('isFraud')['TransactionAmt'].describe().round(2))

## 4. Card Types and Email Domains by Fraud Rate

High-cardinality categoricals (email domain, card network) can be powerful fraud signals if certain values are strongly predictive.

In [ ]:
fig = plot_categorical_fraud_rates(df)
fig.savefig('../evaluation/04_categorical_fraud_rates.png', bbox_inches='tight')
fig.show()

In [ ]:
# Top 15 email domains by volume and fraud rate
email_stats = (
    df.groupby('P_emaildomain', observed=True)['isFraud']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'fraud_rate', 'count': 'n'})
    .query('n >= 500')
    .sort_values('fraud_rate', ascending=False)
    .head(15)
)
email_stats['fraud_rate'] = email_stats['fraud_rate'].map('{:.2%}'.format)
print(email_stats.to_string())

## 5. D-Column Temporal Correlations with isFraud

D-columns encode timedelta features (days since account creation, last login, etc.) and are among the strongest signals for account-takeover fraud.

In [ ]:
fig = plot_d_column_correlations(df)
fig.savefig('../evaluation/05_d_column_correlations.png', bbox_inches='tight')
fig.show()

In [ ]:
# Print ranked correlations with isFraud
import pandas as pd
d_cols = [c for c in df.columns if c.startswith('D')]
corr_with_fraud = df[d_cols + ['isFraud']].corr()['isFraud'].drop('isFraud').sort_values(key=abs, ascending=False)
print('D-column |r| with isFraud (top 15):')
print(corr_with_fraud.head(15).map('{:.4f}'.format).to_string())

---
## Key Findings — Feature Engineering Drivers

1. **Extreme class imbalance (~3.5% fraud):** Accuracy is a meaningless metric. Use AUC-PR as the primary metric throughout. Apply `class_weight='balanced'` or SMOTE for tree models. Set decision thresholds by maximising F1 on a held-out set, not at 0.5.

2. **V-features are massively sparse (up to 95% null) but information-rich:** Do NOT drop them. Use median imputation with a binary `_was_missing` indicator flag for each V-feature. Null pattern itself is predictive — fraudsters often exploit flows that bypass identity verification, leaving V-features empty.

3. **D-columns (timedelta features) are among the strongest fraud signals:** `D1` (days since account creation) and `D4`/`D10` (device/billing history) show the highest |r| with isFraud. New accounts (small D1) transacting at unusual hours are high-risk. Engineer interaction features: `TransactionDT mod 86400` (time-of-day), `TransactionAmt / D1` (velocity by account age).

4. **Email domain and card network carry significant fraud signal:** Several email domains (typically anonymous/free providers) show 10–30× the baseline fraud rate. Encode via target-encoded mean (with leave-one-out to prevent leakage) rather than one-hot — cardinality is high (~60 unique domains). `card4`/`card6` combination (prepaid debit cards) correlates with elevated fraud.

5. **TransactionAmt distribution differs subtly between fraud and legitimate:** Legitimate transactions are concentrated in common retail amounts ($1–$500). Fraud has a heavier tail at very low amounts (card testing, $1–$9) and very high amounts (ATO cashout). Engineer: `log1p(TransactionAmt)`, `TransactionAmt_rounded` (round-number flag), and `TransactionAmt_zscore_by_card1` (per-card deviation from historical mean).